# Assignment 1 Orchestrator

This notebook runs the DWH1 workflow; students do **not** edit its Python cells.  
Prepare `group_NNN.csv` and `csv_load_order_NNN.csv` beside this notebook, and develop SQL in DBeaver.  
The group number comes from the roster filename; CSV load order comes from the 17-row file.

Expected layout:

```text
DWH1_NNN/
    AirQ_DWH1_NNN.ipynb
    csv_load_order_NNN.csv
    group_NNN.csv
    student_bootstrap_NNN.sql
    csv/
    data/
    ddl/
    etl/
    post/
    sqldump/
    src/dwh1_orchestrator.py
```

Student SQL uses plain table names, not group-specific schema prefixes or saved `SET search_path` commands.


## Workflow

1. Validate the external roster and 17-row CSV load order, then display group membership.
2. Preflight required files before changing the database.
3. Connect to the `airq` database using the existing group role.
4. Reset staging; execute staging and extension DDL.
5. Validate foreign-key loading order and load all 17 CSV files **in exactly the provided sequence**.
6. Reset warehouse; execute DWH DDL and SQL-first ETL.
7. Run student validation SQL; create a warehouse-only dump.
8. Show the run summary.


## 1) External configuration — no notebook editing

Before running, rename and complete **both** CSV files next to this notebook:

- `group_NNN.csv` — exactly two completed rows, one for role A and one for role B.
  Enter real student names and student numbers in **both rows** (keep numbers as text).
  For a **one-student group**, enter the **same student number and name** in A and B.
  For a **two-student group**, enter each student's own number and name.
- `csv_load_order_NNN.csv` — exactly 17 rows, one `table_name` per line. Keep the
  15 original table names; add the real names of your two extension tables.
  Reorder the rows so referenced tables load before dependent tables.

`NNN` is your three-digit group number (for example `007`).  
It is read from the roster filename, not entered in Python.  
The notebook checks these files **before** any database reset and never overwrites them.

If a configuration or schema-neutrality check fails, the notebook displays an error and **stops execution** before changing the database. Fix the external file and run again.

In [ ]:
from pathlib import Path
from IPython.display import HTML, Markdown, display
import pandas as pd

from src.dwh1_orchestrator import read_group_roster, read_csv_load_order

ROOT_DIR = Path.cwd()
GROUP_INFO = read_group_roster(ROOT_DIR)
GroupNo = GROUP_INFO["group"]
CSV_LOAD_ORDER = read_csv_load_order(ROOT_DIR, group=GroupNo)

display(Markdown(f"### Group {GroupNo} — {GROUP_INFO['membership']}"))
display(HTML(pd.DataFrame(
    GROUP_INFO["members"],
    columns=["student_role", "student_number", "student_name"],
).to_html(index=False)))
print(f"Validated {len(CSV_LOAD_ORDER)} CSV load-order rows (15 standard + 2 extensions).")


## 2) Setup and preflight

Derive group-specific names and validate all required files and schema-neutral SQL **before any database changes**.

In [ ]:
from src.dwh1_orchestrator import (
    connect_database,
    create_sql_dump,
    database_user,
    evaluate_check_results,
    load_csv_tables,
    post_check_run_status,
    preflight,
    print_run_summary,
    reset_schema,
    run_sql_file,
    run_sql_folder,
    schema_names,
    verify_load_order_dependencies,
)

VERBOSE_SQL = False  # internal diagnostic setting; students do not edit

SCHEMA_STG, SCHEMA_DWH = schema_names(GroupNo)
DB_USER = database_user(GroupNo)
CSV_DIR = ROOT_DIR / "csv"
DATA_DIR = ROOT_DIR / "data"
DDL_DIR = ROOT_DIR / "ddl"
ETL_DIR = ROOT_DIR / "etl"
POST_DIR = ROOT_DIR / "post"
SQLDUMP_DIR = ROOT_DIR / "sqldump"
RUN_STATUS = {}

preflight_info = preflight(ROOT_DIR, group=GroupNo, csv_load_order=CSV_LOAD_ORDER)
RUN_STATUS["Preflight"] = "PASS"


## 3) Database connection

Connect to `airq` using the group role created manually by `student_bootstrap_NNN.sql`.

In [ ]:
engine, DB_SETTINGS = connect_database(
    GroupNo,
    database="airq",
    host="localhost",
    port=5432,
)
RUN_STATUS["Database connection"] = "PASS"


## 4) Reset and create the staging schema

Reset the staging schema and execute the standard and extension DDL files.

In [ ]:
reset_schema(engine, SCHEMA_STG, owner=DB_USER)

run_sql_file(
    DDL_DIR / "airq_create_stg1.sql",
    engine=engine,
    search_path=[SCHEMA_STG],
    progress=VERBOSE_SQL,
)
run_sql_file(
    DDL_DIR / "airq_create_ext.sql",
    engine=engine,
    search_path=[SCHEMA_STG],
    progress=VERBOSE_SQL,
)

RUN_STATUS["Staging schema"] = "PASS"


## 5) Load CSV data into staging

Check the 17-row order against staging foreign keys **before inserting any rows**, then load
all source CSVs in their exact interleaved sequence. `csv/` holds standard tables;
`data/` holds the two student extension tables.

In [ ]:
verify_load_order_dependencies(
    engine,
    schema=SCHEMA_STG,
    csv_load_order=CSV_LOAD_ORDER,
)
RUN_STATUS["CSV order"] = "PASS (17 tables, foreign-key order verified)"

loaded_rows = load_csv_tables(
    CSV_DIR,
    CSV_LOAD_ORDER,
    extension_folder=DATA_DIR,
    engine=engine,
    schema=SCHEMA_STG,
)
RUN_STATUS["CSV load"] = f"PASS ({len(loaded_rows)} tables, {sum(loaded_rows.values()):,} rows)"


## 6) Reset and create the dimensional warehouse

Reset the DWH schema and execute `ddl/airq_create_dwh1.sql`.

In [ ]:
reset_schema(engine, SCHEMA_DWH, owner=DB_USER)

run_sql_file(
    DDL_DIR / "airq_create_dwh1.sql",
    engine=engine,
    search_path=[SCHEMA_DWH],
    progress=VERBOSE_SQL,
)

RUN_STATUS["Warehouse schema"] = "PASS"


## 7) SQL-first ETL

Execute all `etl/a1_etl*.sql` files in lexicographic order. The orchestrator supplies the DWH and staging schemas as the SQL search path.

In [ ]:
etl_results = run_sql_folder(
    ETL_DIR,
    "a1_etl*.sql",
    engine=engine,
    search_path=[SCHEMA_DWH, SCHEMA_STG],
    progress=VERBOSE_SQL,
)

RUN_STATUS["ETL"] = f"PASS ({len(etl_results)} SQL files)"


## 8) Post-ETL student validation

A validation check should test a meaningful assertion about the populated warehouse, not merely execute without an error.

Each `post/a1_check*.sql` file must return one final result set containing exactly one summary row with at least:

- `check_name` — what is being tested;
- `status` — `PASS` or `FAIL`;
- optional diagnostic columns such as counts, mismatches or min/max values.

Use a balanced set of checks covering source-to-target reconciliation, integrity and model-specific semantic correctness.  
Additional checks may cover SCD2 logic, time coverage, measure domains, Table X/Y integration, etc.

In [ ]:
check_results = run_sql_folder(
    POST_DIR,
    "a1_check*.sql",
    engine=engine,
    search_path=[SCHEMA_DWH, SCHEMA_STG],
    progress=VERBOSE_SQL,
)

# Show the diagnostic summary row returned by every student check.
for filename, result in check_results.items():
    print(f"\n{filename}")
    if result is None:
        print("(no result set returned)")
    else:
        display(HTML(result.to_html(index=False)))

# Build a compact summary of the student validation results.
check_summary = evaluate_check_results(check_results)
print("\nStudent validation summary")
display(HTML(check_summary.to_html(index=False)))

RUN_STATUS["Post-ETL student validation"] = post_check_run_status(check_summary)


## 9) Create the warehouse SQL dump

Create a portable dump of the DWH schema only. The helper uses `--no-owner --no-privileges` so the dump can be restored later for grading.

In [ ]:
dump_path = create_sql_dump(
    SQLDUMP_DIR,
    group=GroupNo,
    db_settings=DB_SETTINGS,
    schema=SCHEMA_DWH,
)
RUN_STATUS["SQL dump"] = f"PASS ({dump_path.name})"


## 10) Run summary

In [ ]:
print_run_summary(RUN_STATUS, group=GroupNo)


**Submission reminder:** Keep the completed notebook together with `group_NNN.csv`, `csv_load_order_NNN.csv`,  
original/extension CSV files, SQL DDL, SQL ETL, validation checks,  
warehouse-only SQL dump, ER diagram, and report.  
Students edit external files only, never the notebook or helper.
